<a href="https://colab.research.google.com/github/hadasecohen/streaming-vae-anomaly-detection/blob/main/notebooks/demo_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Streaming VAE Anomaly Detection — GPU Demo

Clones the repo and runs `demo.py` end to end on a free Colab GPU: offline
warmup training + online streaming anomaly detection on a real ERA5 slice.

Runtime menu → Change runtime type → GPU, then run all cells.

**Setup:** this is a private repo, so before running you need a GitHub token as a Colab secret — key icon in the left sidebar -> new secret named `GITHUB_TOKEN`, value = a token from [github.com/settings/tokens](https://github.com/settings/tokens) (read-only `repo` access is enough), then toggle "Notebook access" on.

### Choose which experiment to run

In [ ]:
ANOM = "contextual"    # point | group | contextual
ARCH = "mlp_cyclic"    # mlp | mlp_cyclic | lstm | transformer
BEST = False            # True -> run the finetuned config instead of baseline
GPU = 0                 # GPU index, e.g. 0 -> cuda:0; ignored when ANOM == "point" (always CPU)

print(f"ANOM={ANOM!r}  ARCH={ARCH!r}  BEST={BEST}  GPU={GPU}")

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/hadasecohen/streaming-vae-anomaly-detection.git"
REPO_DIR = Path("/content/repo") if "COLAB_RELEASE_TAG" in os.environ else Path("repo")

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import userdata

    token = userdata.get("GITHUB_TOKEN")
    if not token:
        raise RuntimeError(
            "The Colab secret GITHUB_TOKEN is unavailable. "
            "Open the key icon in the left sidebar, add the token, "
            "and enable Notebook access."
        )

    if not REPO_DIR.exists():
        # Use an askpass helper so the token is not stored in the Git remote URL.
        askpass = Path("/content/git_askpass.sh")
        askpass.write_text(
            '#!/bin/sh\n'
            'case "$1" in\n'
            '  *Username*) echo "x-access-token" ;;\n'
            '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
            'esac\n'
        )
        askpass.chmod(0o700)

        env = os.environ.copy()
        env["GITHUB_TOKEN"] = token
        env["GIT_ASKPASS"] = str(askpass)
        env["GIT_TERMINAL_PROMPT"] = "0"

        try:
            subprocess.run(
                ["git", "clone", REPO_URL, str(REPO_DIR)],
                check=True,
                env=env,
            )
        finally:
            askpass.unlink(missing_ok=True)

    os.chdir(REPO_DIR)
else:
    # When launched from notebooks/ inside a local checkout,
    # move to the repository root.
    if not Path("scripts/run_all_trials.py").exists():
        os.chdir("..")

print("Working directory:", os.getcwd())
print("Installing the project dependencies...")
subprocess.run(
    ["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)
%env MPLBACKEND=Agg
print("Setup complete.")

### Run `demo.py`

Same script you'd run from a terminal — `python demo.py --anom ... --arch ... [--best]`.
See `demo.py --help` for the full option list.

Its output ends with the equivalent `python -m scripts.trial.run_trial <config> <ARCH>` command for this exact experiment, run directly against its tracked config under `standalone_tests/` instead of through `demo.py`. To try that command here, paste it into a new cell with a `!` prefix (e.g. `!python -m scripts.trial.run_trial ...`) — this notebook's working directory is already the cloned repo root, so the printed relative path works as-is.

In [ ]:
_best_flag = "--best" if BEST else ""
!python demo.py --anom {ANOM} --arch {ARCH} {_best_flag} --gpu {GPU}

### Inspect the run

In [ ]:
import yaml

_tuning = "finetuned" if BEST else "baseline"
_arch_dir = {
    "mlp": "MLP", "mlp_cyc": "MLP_Cyclic", "mlp_cyclic": "MLP_Cyclic",
    "lstm": "LSTM", "tr": "Transformer", "tf": "Transformer",
    "transformer": "Transformer", "transtormer": "Transformer",
}[ARCH]
_config_path = f"standalone_tests/{_tuning}/era5_{ANOM}_{_arch_dir}_config_standalone.yaml"
with open(_config_path) as f:
    _cfg = yaml.safe_load(f)

RUN_DIR = _cfg["common"]["logging"]["run_dir"]
TEST_PATH = _cfg["data"]["test_path"]

# The METRICS / THRESHOLD TUNING block at the end of run.log is the run's own summary
with open(f"{RUN_DIR}/run.log") as f:
    lines = f.readlines()
print("".join(lines[-40:]))

### Score histogram (normal vs. anomaly)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

preds = pd.read_csv(
    f"{RUN_DIR}/trial_predictions.csv", skiprows=1, header=None,
    names=["ts_start", "ts_end", "step", "confusion", "label",
           "final_pred", "trained", "final_conf", "metrics"],
)

fig, ax = plt.subplots(figsize=(8, 4))
for lbl, color, name in [(0, "#27ae60", "normal"), (1, "#e74c3c", "anomaly")]:
    sub = preds.loc[preds["label"] == lbl, "final_conf"].dropna()
    if len(sub):
        ax.hist(sub, bins=80, alpha=0.55, color=color, density=True, label=name)
ax.set_xlabel("final confidence score")
ax.set_ylabel("density")
ax.set_title(f"Score distribution — {ARCH} — {ANOM} scenario")
ax.legend()

# Clip y-axis when a saturation spike (e.g. many scores pinned near 0 or 1)
# dominates the scale: if the tallest bar is > 3x the 90th-percentile bar,
# cap at 2x p90 so the bulk of the distribution stays visible (matches
# scripts/cross_compare.py's score-histogram clipping).
heights = sorted(p.get_height() for p in ax.patches if p.get_height() > 0)
if len(heights) >= 4:
    p90 = float(np.percentile(heights, 90))
    if heights[-1] > 3.0 * p90 > 0:
        ax.set_ylim(0, p90 * 2.0)
        ax.annotate("▲ clipped", xy=(1, 1), xycoords="axes fraction",
                    fontsize=8, color="gray", ha="right", va="top")

plt.show()

print(
    "Better separation between the green (normal) and red (anomaly) "
    "distributions means the model's score more reliably distinguishes "
    "anomalies from normal observations."
)

### False-positive rate over time

In [ ]:
preds["is_pos"] = preds["confusion"].isin(["TP", "FN"])
preds["is_neg"] = preds["confusion"].isin(["TN", "FP"])
preds["is_fp"]  = preds["confusion"] == "FP"
preds["is_tn"]  = preds["confusion"] == "TN"

# Rolling FPR = FP / (FP + TN) over a sliding window of steps (same formula
# as scripts/plot_fpr_over_time.py's rolling_metric, single-run/no seed averaging).
WINDOW = max(200, len(preds) // 50)
roll_fp = preds["is_fp"].rolling(window=WINDOW, min_periods=1).sum()
roll_tn = preds["is_tn"].rolling(window=WINDOW, min_periods=1).sum()
roll_fpr = (roll_fp / (roll_fp + roll_tn)).where((roll_fp + roll_tn) > 0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={"width_ratios": [2, 1]})

ax = axes[0]
ax.plot(preds["step"], roll_fpr, lw=1.2, color="#e74c3c")
ax.set_xlabel("stream step")
ax.set_ylabel("rolling FPR")
ax.set_title(f"Rolling FPR (window={WINDOW}) — {ARCH} — {ANOM} scenario")
ax.set_ylim(bottom=0)

# Bucket display: split into 5 equal-anomaly-count buckets (matches
# scripts/plot_fpr_over_time.py's get_anomaly_cuts/bucket_metric) plus an
# overall column, and show FPR per bucket as a bar chart.
N_BUCKETS = 5
pos_idx = preds.index[preds["is_pos"]].to_numpy()
ax = axes[1]
if len(pos_idx) >= N_BUCKETS:
    cuts = [0] + [int(pos_idx[int(i * len(pos_idx) / N_BUCKETS)]) for i in range(1, N_BUCKETS)] + [len(preds)]
    bucket_labels = [f"{i+1}" for i in range(N_BUCKETS)] + ["all"]
    bucket_fpr = []
    for i in range(N_BUCKETS):
        seg = preds.iloc[cuts[i]:cuts[i + 1]]
        fp, tn = seg["is_fp"].sum(), seg["is_tn"].sum()
        bucket_fpr.append(fp / (fp + tn) if (fp + tn) > 0 else float("nan"))
    fp, tn = preds["is_fp"].sum(), preds["is_tn"].sum()
    bucket_fpr.append(fp / (fp + tn) if (fp + tn) > 0 else float("nan"))
    ax.bar(bucket_labels, bucket_fpr, color="#e74c3c", alpha=0.75)
    ax.set_xlabel("bucket (equal anomaly count) / all")
    ax.set_ylabel("FPR")
    ax.set_title("FPR by bucket")
else:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

print(
    "A rising rolling FPR points to miscalibration developing as the stream "
    "progresses (e.g. an early-burst effect or threshold drift); the bucket "
    "chart shows whether false positives concentrate in specific parts of "
    "the stream rather than spreading evenly."
)

### Confusion counts per metric

In [ ]:
from IPython.display import Image, display

display(Image(filename=f"{RUN_DIR}/confusion_histograms.png"))

TP/TN counts (green) vs. FN/FP counts (red) for each metric that
contributes to the final score — shows which metrics best separate
normal from anomalous observations, and where errors are concentrated.